In [2]:
from pyspark.sql import SparkSession, functions as F

spark = (
    SparkSession.builder
    .appName("MinIO-PostgreSQL")
    .master("local[*]")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .config("spark.jars", "/home/jovyan/postgresql-42.7.1.jar")  # PostgreSQL JDBC driver
    .getOrCreate()
)

hconf = spark.sparkContext._jsc.hadoopConfiguration()
hconf.set("fs.s3a.endpoint", "http://minio:9000")
hconf.set("fs.s3a.access.key", "matrix")
hconf.set("fs.s3a.secret.key", "matrix123")
hconf.set("fs.s3a.path.style.access", "true")
hconf.set("fs.s3a.connection.ssl.enabled", "false")
hconf.set("fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")

In [3]:
bronze_path = "s3a://bronze/"
silver_path = "s3a://silver/"
gold_path = "s3a://gold/"

In [4]:
import os
file_path = "/home/jovyan/work"
files = os.listdir(file_path)
print(files)

['docker-compose.yaml', 'spark-defaults.conf', 'notebooks', 'output.txt', 'Untitled(7)(3)(1).ipynb', 'spark1.ipynb', 'spark1.bash', 'card_trn.csv', 'cust.csv', 'spark2.ipynb', 'task2_transacion.csv']


In [5]:
df_all = spark.read.csv(
    "/home/jovyan/work/task2_transacion.csv",
    header=True,
    inferSchema=True
)

In [6]:
print(df_all.count())

df_all.printSchema()
df_all.show(10, truncate=False)

100
root
 |-- transaction_id: integer (nullable = true)
 |-- card_number: string (nullable = true)
 |-- mcc: integer (nullable = true)
 |-- amount: double (nullable = true)
 |-- currency: string (nullable = true)
 |-- dr_cr: string (nullable = true)
 |-- transaction_date: timestamp (nullable = true)
 |-- merchant_name: string (nullable = true)
 |-- merchant_city: string (nullable = true)
 |-- channel: string (nullable = true)
 |-- status: string (nullable = true)

+--------------+------------+----+-------+--------+-----+-------------------+-------------+-------------+-------+--------+
|transaction_id|card_number |mcc |amount |currency|dr_cr|transaction_date   |merchant_name|merchant_city|channel|status  |
+--------------+------------+----+-------+--------+-----+-------------------+-------------+-------------+-------+--------+
|1             |4000****2824|5411|1112.58|AZN     |D    |2025-12-19 01:26:56|Amazon       |Baku         |ATM    |REVERSED|
|2             |4000****2424|5942|633.4

In [7]:
df_part1 = df_all.limit(50)

In [8]:
df_part2 = df_all.subtract(df_part1)

In [9]:
df_part1.coalesce(1).write.mode("overwrite").csv("/home/jovyan/work/part1", header=True)
df_part2.coalesce(1).write.mode("overwrite").csv("/home/jovyan/work/part2", header=True)

In [10]:
from delta.tables import DeltaTable

bronze_delta_path = "s3a://bronze/transactions_delta"

In [11]:
df_part1_reload = spark.read.csv("/home/jovyan/work/part1", header=True, inferSchema=True)

df_part1_reload.write.format("delta") \
    .mode("overwrite") \
    .partitionBy("transaction_date") \
    .save(bronze_delta_path)

In [12]:
delta_table = DeltaTable.forPath(spark, bronze_delta_path)

In [13]:
schema_info = delta_table.toDF().schema

In [18]:
for field in schema_info.fields:
    print(f"{field.name}: {field.dataType}")

transaction_id: IntegerType()
card_number: StringType()
mcc: IntegerType()
amount: DoubleType()
currency: StringType()
dr_cr: StringType()
transaction_date: TimestampType()
merchant_name: StringType()
merchant_city: StringType()
channel: StringType()
status: StringType()


In [19]:
partition_columns = delta_table.detail().select("partitionColumns").collect()[0][0]

In [20]:
print(partition_columns)

['transaction_date']


In [21]:
df_part2_raw = spark.read.csv("/home/jovyan/work/part2", header=True, inferSchema=False)

In [23]:
df_part2_raw.printSchema()

root
 |-- transaction_id: string (nullable = true)
 |-- card_number: string (nullable = true)
 |-- mcc: string (nullable = true)
 |-- amount: string (nullable = true)
 |-- currency: string (nullable = true)
 |-- dr_cr: string (nullable = true)
 |-- transaction_date: string (nullable = true)
 |-- merchant_name: string (nullable = true)
 |-- merchant_city: string (nullable = true)
 |-- channel: string (nullable = true)
 |-- status: string (nullable = true)



In [24]:
from pyspark.sql.functions import col

df_part2_typed = df_part2_raw.select(
    *[col(field.name).cast(field.dataType).alias(field.name) for field in schema_info.fields]
)

In [25]:
df_part2_typed.printSchema()

root
 |-- transaction_id: integer (nullable = true)
 |-- card_number: string (nullable = true)
 |-- mcc: integer (nullable = true)
 |-- amount: double (nullable = true)
 |-- currency: string (nullable = true)
 |-- dr_cr: string (nullable = true)
 |-- transaction_date: timestamp (nullable = true)
 |-- merchant_name: string (nullable = true)
 |-- merchant_city: string (nullable = true)
 |-- channel: string (nullable = true)
 |-- status: string (nullable = true)



In [26]:
df_part2_typed.write.format("delta") \
    .mode("append") \
    .partitionBy(*partition_columns) \
    .save(bronze_delta_path)

In [27]:
df_final = spark.read.format("delta").load(bronze_delta_path)

In [28]:
df_final.count()

100

In [29]:
df_final.show(5, truncate=False)

+--------------+------------+----+-------+--------+-----+-------------------+-------------+-------------+-------+-------+
|transaction_id|card_number |mcc |amount |currency|dr_cr|transaction_date   |merchant_name|merchant_city|channel|status |
+--------------+------------+----+-------+--------+-----+-------------------+-------------+-------------+-------+-------+
|66            |4000****2124|5411|1131.6 |AZN     |D    |2025-12-18 02:01:56|Bravo        |Lankaran     |POS    |SUCCESS|
|11            |4000****5339|5999|370.69 |EUR     |C    |2026-02-02 06:10:56|McDonalds    |Shaki        |ATM    |FAILED |
|68            |4000****2876|5999|1447.12|AZN     |D    |2025-12-21 23:43:56|Bolt         |Lankaran     |ATM    |FAILED |
|48            |4000****5920|5541|464.62 |USD     |C    |2025-12-11 17:57:56|Zara         |Ganja        |ATM    |SUCCESS|
|59            |4000****3471|5732|575.17 |AZN     |D    |2026-02-02 16:00:56|Araz         |Shaki        |ECOM   |FAILED |
+--------------+--------